# **ТИПиС - ИДЗ№5 - Воропаев Илья 3391**



In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('jamb_exam_results.csv')

In [3]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

**Подготовка**
-- --

In [4]:
df = df.drop('student_id', axis=1)
df = df.fillna(0)

In [7]:
y = df.jamb_score.values
X = df.drop('jamb_score', axis=1)
df_full_train, df_test, y_full_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)
df_train, df_val, y_train, y_val = train_test_split(
    df_full_train, y_full_train, test_size=0.25, random_state=1
)

In [8]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

**Вопрос 1**
-- --

In [10]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

feature = dv.feature_names_[dt.tree_.feature[0]]
print(feature)

study_hours_per_week


Ответ: study_hours_per_week

**Вопрос 2**
-- --

In [13]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

42.13724207871227


Ответ: 42.13

**Вопрос 3**
-- --

In [16]:
results = []
for n in range(10, 210, 10):
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    results.append((n, rmse))
df_scores = pd.DataFrame(results, columns=['n_estimators', 'rmse'])

In [24]:
# Анализируем улучшения
print("Анализ улучшений RMSE:")
for i in range(1, len(df_scores)):
    improvement = df_scores.loc[i-1, 'rmse'] - df_scores.loc[i, 'rmse']
    print(f"n_estimators {df_scores.loc[i-1, 'n_estimators']}->{df_scores.loc[i, 'n_estimators']}: "
          f"Улучшение = {improvement:.3f}")

Анализ улучшений RMSE:
n_estimators 10->20: Улучшение = 0.676
n_estimators 20->30: Улучшение = 0.355
n_estimators 30->40: Улучшение = 0.189
n_estimators 40->50: Улучшение = 0.065
n_estimators 50->60: Улучшение = 0.068
n_estimators 60->70: Улучшение = 0.107
n_estimators 70->80: Улучшение = 0.138
n_estimators 80->90: Улучшение = 0.035
n_estimators 90->100: Улучшение = -0.012
n_estimators 100->110: Улучшение = -0.077
n_estimators 110->120: Улучшение = -0.031
n_estimators 120->130: Улучшение = -0.026
n_estimators 130->140: Улучшение = 0.056
n_estimators 140->150: Улучшение = -0.002
n_estimators 150->160: Улучшение = -0.007
n_estimators 160->170: Улучшение = -0.024
n_estimators 170->180: Улучшение = -0.014
n_estimators 180->190: Улучшение = 0.010
n_estimators 190->200: Улучшение = 0.030


RMSE перестаёт уменьшаться после значения
параметра 80

Ответ: 80

**Вопрос 4**
-- --

In [27]:
depth_results = []

for depth_val in [10, 15, 20, 25]:
    rmse_list = []

    for n_trees in range(10, 201, 10):
        model = RandomForestRegressor(
            max_depth=depth_val,
            n_estimators=n_trees,
            random_state=1,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_list.append(rmse)

    depth_results.append((depth_val, np.mean(rmse_list)))

best_depth = min(depth_results, key=lambda x: x[1])[0]
print(best_depth)

10


Ответ: 10

**Вопрос 5**
-- --

In [28]:
rf_final = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_final.fit(X_train, y_train)

importances = pd.Series(rf_final.feature_importances_, index=dv.feature_names_)
importances = importances.sort_values(ascending=False)

target_features = ['study_hours_per_week', 'attendance_rate', 'distance_to_school', 'teacher_quality']
most_important = None
for feature in importances.index:
    if feature in target_features:
        most_important = feature
        break

print(most_important)

study_hours_per_week


Ответ: study_hours_per_week

**Итоговые ответы**
-- --

1. study_hours_per_week
2. 42.13
3. 80
4. 10
5. study_hours_per_week